# 라벨 기반 추출 재현율 점검

합성 좌표와 합성 그래프 CSV로 `label_eval`을 점검합니다. 원문 PDF 텍스트는 포함하지 않습니다.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import tempfile

WORK = Path(tempfile.mkdtemp())
DATA_ROOT = WORK / 'data'
LABELS = WORK / 'labels.csv'
POOL = WORK / 'pool.json'
GRAPH_DIR = WORK / 'graph'
OUTPUT_DIR = WORK / 'output'
(DATA_ROOT / 'pdf').mkdir(parents=True)
GRAPH_DIR.mkdir()


## 합성 입력 예시

In [ ]:
pdf = DATA_ROOT / 'pdf' / 'synthetic.pdf'
pdf.write_bytes(b'synthetic-pdf')
sha = hashlib.sha256(pdf.read_bytes()).hexdigest()
with LABELS.open('w', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['id', 'issuer', 'pid', 'kind', 'text', 'start', 'end', 'parent_id', 'relation'])
    writer.writeheader()
    writer.writerow({'id': 'b1', 'issuer': 'demo', 'pid': 'p1', 'kind': 'Benefit', 'text': 'synthetic', 'start': 5, 'end': 10, 'parent_id': '', 'relation': ''})
    writer.writerow({'id': 'l1', 'issuer': 'demo', 'pid': 'p1', 'kind': 'Limit', 'text': 'synthetic', 'start': 81, 'end': 90, 'parent_id': '', 'relation': ''})
with POOL.open('w', encoding='utf-8') as stream:
    json.dump([{'pid': 'p1', 'issuer': 'demo', 'path': 'pdf/synthetic.pdf', 'page': 1, 'text': 'synthetic'}], stream)
def write_csv(name, columns, rows):
    with (GRAPH_DIR / name).open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=columns); writer.writeheader(); writer.writerows(rows)
node_id = f'{sha[:16]}-p0001-c001:b1'
write_csv('fp_nodes.csv', ['id', 'node_type', 'props', 'chunk_id', 'run_id'], [{'id': node_id, 'node_type': 'Benefit', 'props': '{}', 'chunk_id': 'synthetic', 'run_id': 'r'}])
write_csv('fp_edges.csv', ['id', 'source_id', 'target_id', 'edge_type', 'run_id'], [])
write_csv('fp_evidence.csv', ['node_id', 'edge_id', 'doc_sha256', 'page_no', 'start_pos', 'end_pos', 'quote_status'], [{'node_id': node_id, 'edge_id': '', 'doc_sha256': sha, 'page_no': 1, 'start_pos': 5, 'end_pos': 10, 'quote_status': 'located'}, {'node_id': limit_node_id, 'edge_id': '', 'doc_sha256': sha, 'page_no': 1, 'start_pos': 60, 'end_pos': 70, 'quote_status': 'located'}])


## 모듈 호출과 결과 표시

In [ ]:
from payment_card.file_pipeline.extraction.label_eval import evaluate

summary = evaluate(LABELS, POOL, [f'demo={GRAPH_DIR}'], DATA_ROOT, OUTPUT_DIR, overlap='any')
summary['labels']['all']['by_kind']['Benefit'], list(csv.DictReader((OUTPUT_DIR / 'misses.csv').open(encoding='utf-8')))


## 새 미스 분류 필드 확인

In [ ]:
miss_rows = list(csv.DictReader((OUTPUT_DIR / 'misses.csv').open(encoding='utf-8')))
assert {'detail', 'other_kind', 'nearest_gap'} <= set(miss_rows[0])
assert miss_rows[0]['detail'] == 'boundary_near'
assert summary['near_hits']['Limit'] == 1
summary['miss_detail_counts']['boundary_near']['Limit'], miss_rows[0]
